# 04 — Streaming in Agent Loops: Examples

> Build an agent that streams its reasoning AND calls tools mid-response.

**What you'll build:**
1. Basic streaming agent step
2. Tool call detection + execution during stream
3. Full multi-step streaming agent loop
4. Pretty streaming agent output with Rich

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint
from rich.console import Console

load_dotenv()
client = OpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: Define Tools & Executors

In [ ]:
# Tool schema definitions
TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string"}
        }, "required": ["city"]}
    }},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate a math expression",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string"}
        }, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "search",
        "description": "Search the web for current information",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}
        }, "required": ["query"]}
    }},
]

# Mock tool executors
def get_weather(city: str) -> str:
    return f"Weather in {city}: 22°C, partly cloudy, humidity 65%"

def calculate(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

def search(query: str) -> str:
    return f"Search results for '{query}': [Mock result 1] [Mock result 2]"

TOOL_EXECUTOR = {"get_weather": get_weather, "calculate": calculate, "search": search}
print('✅ Tools defined')

---
## Part 2: Single Streaming Step with Tool Detection

In [ ]:
def streaming_step(messages: list[dict]) -> dict:
    """
    Execute one streaming step of the agent loop.
    Returns: {type: 'stop'|'tool_calls', text, tool_calls}
    """
    full_text = ""
    tool_calls_acc = {}
    finish_reason = None

    rprint("[bold cyan]🤖 Agent:[/bold cyan] ", end="")

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
        stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        finish_reason = chunk.choices[0].finish_reason or finish_reason

        if delta.content:
            full_text += delta.content
            print(delta.content, end="", flush=True)

        if delta.tool_calls:
            for tc in delta.tool_calls:
                idx = tc.index
                if idx not in tool_calls_acc:
                    tool_calls_acc[idx] = {"id": "", "name": "", "arguments": ""}
                if tc.id:                       tool_calls_acc[idx]["id"]        += tc.id
                if tc.function.name:            tool_calls_acc[idx]["name"]      += tc.function.name
                if tc.function.arguments:       tool_calls_acc[idx]["arguments"] += tc.function.arguments

    print()
    return {
        "type": finish_reason or "stop",
        "text": full_text,
        "tool_calls": [{**tc, "args": json.loads(tc["arguments"] or "{}")} for tc in tool_calls_acc.values()]
    }


# Test: a prompt that should trigger a tool call
print("📤 Request: 'What is the weather in Paris?'\n")
messages = [
    {"role": "system", "content": "You are a helpful assistant with tool access."},
    {"role": "user", "content": "What is the weather in Paris?"}
]
step_result = streaming_step(messages)
rprint(f"\n[bold]finish_reason:[/bold] {step_result['type']}")
if step_result['tool_calls']:
    for tc in step_result['tool_calls']:
        rprint(f"[yellow]🔧 Tool call: {tc['name']}({tc['args']})[/yellow]")

---
## Part 3: Full Multi-Step Streaming Agent Loop

In [ ]:
def run_streaming_agent(
    user_message: str,
    max_steps: int = 6,
) -> str:
    """Full streaming agent loop with tool execution."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools to answer questions accurately."},
        {"role": "user", "content": user_message}
    ]

    rprint(f"\n[bold]🚀 Agent starting:[/bold] {user_message}")
    rprint("─" * 60)

    for step in range(max_steps):
        rprint(f"\n[bold white]Step {step+1}:[/bold white]")

        result = streaming_step(messages)

        if result["type"] == "stop":
            rprint("\n[bold green]✅ Agent finished[/bold green]")
            return result["text"]

        elif result["type"] == "tool_calls":
            # Add assistant's response to history
            tc_list = [{
                "id":   tc["id"],
                "type": "function",
                "function": {"name": tc["name"], "arguments": tc["arguments"]}
            } for tc in result["tool_calls"]]

            messages.append({
                "role": "assistant",
                "content": result["text"] or None,
                "tool_calls": tc_list
            })

            # Execute each tool
            for tc in result["tool_calls"]:
                rprint(f"   [yellow]🔧 {tc['name']}({tc['args']})[/yellow]")
                executor = TOOL_EXECUTOR.get(tc["name"])
                tool_result = executor(**tc["args"]) if executor else f"Tool not found: {tc['name']}"
                rprint(f"   [green]📋 {tool_result}[/green]")

                messages.append({
                    "role":        "tool",
                    "tool_call_id": tc["id"],
                    "content":     str(tool_result)
                })

    return "Max steps reached."


# ── Test the full agent ────────────────────────────────────────────────────
final = run_streaming_agent(
    "What is the weather in Tokyo, and what is 15% of the temperature in Celsius?"
)
rprint(f"\n[bold]Final answer:[/bold] {final}")

---
## Summary

| Component | Purpose |
|---|---|
| `streaming_step()` | One streaming LLM call; detects finish_reason |
| Tool accumulation | Collect `delta.tool_calls[i].function.arguments` across chunks |
| History management | Append assistant + tool messages correctly |
| `run_streaming_agent()` | Full loop: stream → detect → execute → continue |

**Next:** `05_streaming_tool_calls` — deep dive into partial JSON assembly for tool calls.